[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session5/student/Lab5_MultiAgent_HITL_Student.ipynb)

# Lab 5: Multi-Agent Patterns & Human-in-the-Loop
## CCI Session 5

**Duration:** 15 minutes

### Clinical Scenario
> Build a clinical decision support system with specialized agents: a triage agent classifies the case, routes to an oncology specialist or general medicine agent, and any medication change requires human (physician) approval before execution.

### Objective
- Build a multi-agent system with orchestrator-worker pattern
- Implement human-in-the-loop with `interrupt()`
- Handle agent handoffs between specialists
- Add safety guardrails for clinical decisions

---
## Setup

In [ ]:
# Install required packages
!pip install -q langchain langchain-openai langgraph

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from typing import Literal

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

---
## Cell 1 -- Define Specialist Tools

Each specialist agent has its own set of clinical tools.

In [ ]:
# ---- Oncology Specialist Tools ----

# TODO: Create an oncology staging tool
# @tool
# def check_cancer_staging(cancer_type: str, stage: str) -> str:
#     """Look up cancer staging details and prognosis information."""
#     staging_db = {
#         ("NSCLC", "IIIA"): "Stage IIIA NSCLC: Locally advanced. 5-year survival ~36%. Treatment: concurrent chemoradiation or surgery + adjuvant chemo. Consider immunotherapy (pembrolizumab) if PD-L1 >= 50%.",
#         ("breast", "IIA"): "Stage IIA Breast Cancer: T2N0 or T1N1. 5-year survival ~93%. Treatment: surgery + adjuvant chemo (AC-T) +/- radiation. Consider HER2 and hormone receptor status.",
#         ("CRC", "III"): "Stage III CRC: Node-positive. 5-year survival ~71%. Treatment: surgery + FOLFOX adjuvant chemo for 3-6 months.",
#     }
#     key = (cancer_type, stage)
#     return staging_db.get(key, f"No staging data found for {cancer_type} stage {stage}. Consult tumor board.")

# TODO: Create a chemo protocol lookup tool
# @tool
# def lookup_chemo_protocol(protocol_name: str) -> str:
#     """Look up chemotherapy protocol details including drugs, dosing, and schedule."""
#     protocols = {
#         "carboplatin/paclitaxel": "Carboplatin AUC 6 + Paclitaxel 200mg/m2 IV q21d x 4-6 cycles. Pre-med: dexamethasone, diphenhydramine, famotidine.",
#         "AC-T": "Doxorubicin 60mg/m2 + Cyclophosphamide 600mg/m2 q21d x 4, then Paclitaxel 80mg/m2 weekly x 12. Monitor cardiac function (LVEF).",
#         "FOLFOX": "Oxaliplatin 85mg/m2 + Leucovorin 400mg/m2 + 5-FU 400mg/m2 bolus then 2400mg/m2 over 46h. q14d x 12 cycles.",
#         "pembrolizumab": "Pembrolizumab 200mg IV q21d. Requires PD-L1 testing. Monitor for immune-related adverse events.",
#     }
#     return protocols.get(protocol_name, f"Protocol '{protocol_name}' not found. Check NCCN guidelines.")

# TODO: Create a drug interaction checker
# @tool
# def check_drug_interaction(drug1: str, drug2: str) -> str:
#     """Check for interactions between two drugs."""
#     interactions = {
#         ("carboplatin", "pembrolizumab"): "SAFE: No significant interaction. Combination used in KEYNOTE-189.",
#         ("paclitaxel", "ondansetron"): "CAUTION: Both can prolong QT interval. Monitor ECG.",
#         ("doxorubicin", "trastuzumab"): "WARNING: Combined cardiotoxicity. Monitor LVEF every 3 months.",
#     }
#     key = tuple(sorted([drug1.lower(), drug2.lower()]))
#     return interactions.get(key, f"No known interaction data for {drug1} + {drug2}. Verify with pharmacy.")

# ---- General Medicine Tools ----

# TODO: Create general medicine tools
# @tool
# def check_vitals_history(patient_mrn: str) -> str:
#     """Retrieve recent vital signs for a patient."""
#     vitals = {
#         "18887": "BP 145/92, HR 78, Temp 37.1C, SpO2 96%, Weight 74kg. Trend: BP elevated over last 3 visits.",
#         "21307": "BP 118/76, HR 82, Temp 36.8C, SpO2 99%, Weight 61kg. All vitals stable.",
#     }
#     return vitals.get(patient_mrn, f"No vitals found for MRN {patient_mrn}")

# @tool
# def lookup_medications(patient_mrn: str) -> str:
#     """Look up current medications for a patient."""
#     meds = {
#         "18887": "Current meds: Amlodipine 5mg daily, Metformin 1000mg BID, Omeprazole 20mg daily, Carboplatin/Paclitaxel (cycle 3).",
#         "21307": "Current meds: Tamoxifen 20mg daily, Calcium + Vitamin D, AC-T chemo (cycle 2 of doxorubicin).",
#     }
#     return meds.get(patient_mrn, f"No medication record for MRN {patient_mrn}")

# @tool
# def check_allergies(patient_mrn: str) -> str:
#     """Check patient allergy records."""
#     allergies = {
#         "18887": "ALLERGIES: Penicillin (anaphylaxis), Sulfa drugs (rash). No food allergies.",
#         "21307": "ALLERGIES: None known (NKDA).",
#     }
#     return allergies.get(patient_mrn, f"No allergy record for MRN {patient_mrn}")

print("Tools defined. Uncomment the code above to activate them.")

---
## Cell 2 -- Build the Orchestrator Graph

The orchestrator:
1. **Triage** -- classifies the case
2. **Routes** -- sends to the right specialist
3. **Review** -- pauses for human approval on medication changes

In [ ]:
# TODO: Create the specialist agents
# oncology_tools = [check_cancer_staging, lookup_chemo_protocol, check_drug_interaction]
# general_tools = [check_vitals_history, lookup_medications, check_allergies]

# TODO: Define the triage node -- classifies the case
# def triage_node(state: MessagesState) -> Command[Literal["oncology_node", "general_node"]]:
#     """Classify the case and route to the appropriate specialist."""
#     system_msg = {
#         "role": "system",
#         "content": (
#             "You are a clinical triage agent at KHCC. Analyze the case and decide:\n"
#             "- If it involves cancer diagnosis, staging, chemotherapy, or oncology treatment -> respond with EXACTLY 'ONCOLOGY'\n"
#             "- If it involves general medicine, vitals, non-cancer medications -> respond with EXACTLY 'GENERAL'\n"
#             "Respond with only one word: ONCOLOGY or GENERAL"
#         )
#     }
#     messages = [system_msg] + state["messages"]
#     response = model.invoke(messages)
#     triage_decision = response.content.strip().upper()
#
#     if "ONCOLOGY" in triage_decision:
#         goto = "oncology_node"
#     else:
#         goto = "general_node"
#
#     return Command(
#         goto=goto,
#         update={"messages": [{"role": "assistant", "content": f"[Triage] Routing to {goto.replace('_node', '')} specialist."}]}
#     )

# TODO: Define oncology specialist node
# def oncology_node(state: MessagesState):
#     """Oncology specialist agent processes the case."""
#     oncology_agent = create_react_agent(
#         model=model,
#         tools=oncology_tools,
#         prompt="You are an oncology specialist at KHCC. Analyze the case, check staging, protocols, and drug interactions. If you recommend ANY medication change, clearly state 'MEDICATION CHANGE RECOMMENDED' in your response."
#     )
#     result = oncology_agent.invoke({"messages": state["messages"]})
#     last_msg = result["messages"][-1].content
#
#     # Check if medication change was recommended
#     if "MEDICATION CHANGE" in last_msg.upper():
#         return Command(
#             goto="review_node",
#             update={"messages": result["messages"][len(state["messages"]):]}
#         )
#     return {"messages": result["messages"][len(state["messages"]):] }

# TODO: Define general medicine node
# def general_node(state: MessagesState):
#     """General medicine agent processes the case."""
#     general_agent = create_react_agent(
#         model=model,
#         tools=general_tools,
#         prompt="You are a general medicine physician at KHCC. Review vitals, medications, and allergies. If you recommend ANY medication change, clearly state 'MEDICATION CHANGE RECOMMENDED' in your response."
#     )
#     result = general_agent.invoke({"messages": state["messages"]})
#     last_msg = result["messages"][-1].content
#
#     if "MEDICATION CHANGE" in last_msg.upper():
#         return Command(
#             goto="review_node",
#             update={"messages": result["messages"][len(state["messages"]):] }
#         )
#     return {"messages": result["messages"][len(state["messages"]):] }

print("Node functions defined. Uncomment the code above to activate them.")

---
## Cell 3 -- Add Human-in-the-Loop with `interrupt()`

When a medication change is recommended, the graph **pauses** and waits for a physician to approve or reject.

In [ ]:
# TODO: Define the human review node using interrupt()
# def review_node(state: MessagesState):
#     """Pause execution and wait for physician approval."""
#     last_recommendation = state["messages"][-1].content
#
#     # interrupt() pauses the graph and returns control to the caller
#     # The value passed to interrupt() is what the caller sees
#     human_decision = interrupt(
#         {
#             "type": "medication_review",
#             "recommendation": last_recommendation,
#             "prompt": "PHYSICIAN REVIEW REQUIRED: Please approve or reject this medication change.",
#             "options": ["approve", "reject", "modify"]
#         }
#     )
#
#     # When the graph resumes, human_decision contains the physician's response
#     if human_decision.get("action") == "approve":
#         return {
#             "messages": [{"role": "assistant", "content": f"APPROVED by physician: {human_decision.get('note', 'No additional notes')}. Proceeding with medication change."}]
#         }
#     elif human_decision.get("action") == "reject":
#         return {
#             "messages": [{"role": "assistant", "content": f"REJECTED by physician: {human_decision.get('reason', 'No reason given')}. Medication change will NOT be made."}]
#         }
#     else:
#         return {
#             "messages": [{"role": "assistant", "content": f"MODIFIED by physician: {human_decision.get('modification', 'See notes')}."}]
#         }

# TODO: Build the full StateGraph
# builder = StateGraph(MessagesState)
#
# builder.add_node("triage_node", triage_node)
# builder.add_node("oncology_node", oncology_node)
# builder.add_node("general_node", general_node)
# builder.add_node("review_node", review_node)
#
# builder.add_edge(START, "triage_node")
# # triage_node uses Command to route, so no explicit edges needed from it
# builder.add_edge("oncology_node", END)
# builder.add_edge("general_node", END)
# builder.add_edge("review_node", END)
#
# checkpointer = MemorySaver()
# graph = builder.compile(checkpointer=checkpointer)

print("Review node and graph defined. Uncomment the code above to activate.")

---
## Cell 4 -- Run the Full System

Test with three clinical cases. Notice how the graph routes differently and pauses for approval when needed.

In [ ]:
# TODO: Test Case 1 -- Oncology case with medication change
# config1 = {"configurable": {"thread_id": "case-001"}}
# result1 = graph.invoke(
#     {"messages": [{"role": "user", "content": "Patient MRN 18887, Stage IIIA NSCLC on carboplatin/paclitaxel. PD-L1 is 60%. Should we add pembrolizumab immunotherapy?"}]},
#     config=config1
# )
# print("=" * 60)
# print("CASE 1: Oncology - Adding Immunotherapy")
# print("=" * 60)
# for msg in result1["messages"]:
#     print(f"[{msg.type}]: {msg.content[:200]}")
#     print()

# If the graph paused for review, resume with physician approval:
# result1_resumed = graph.invoke(
#     Command(resume={"action": "approve", "note": "Approved. Start pembrolizumab 200mg q21d. Schedule PFTs baseline."}),
#     config=config1
# )
# print("After physician approval:")
# print(result1_resumed["messages"][-1].content)

print("Uncomment the code above to run Case 1.")

In [ ]:
# TODO: Test Case 2 -- General medicine with medication adjustment
# config2 = {"configurable": {"thread_id": "case-002"}}
# result2 = graph.invoke(
#     {"messages": [{"role": "user", "content": "Patient MRN 18887 has persistent hypertension (BP 145/92 over last 3 visits). Currently on Amlodipine 5mg. Needs medication adjustment."}]},
#     config=config2
# )
# print("=" * 60)
# print("CASE 2: General Medicine - Hypertension Management")
# print("=" * 60)
# for msg in result2["messages"]:
#     print(f"[{msg.type}]: {msg.content[:200]}")
#     print()

# If paused, resume with physician decision:
# result2_resumed = graph.invoke(
#     Command(resume={"action": "modify", "modification": "Increase Amlodipine to 10mg daily instead of adding new drug. Recheck BP in 2 weeks."}),
#     config=config2
# )
# print("After physician modification:")
# print(result2_resumed["messages"][-1].content)

print("Uncomment the code above to run Case 2.")

In [ ]:
# TODO: Test Case 3 -- Simple case, no medication change needed
# config3 = {"configurable": {"thread_id": "case-003"}}
# result3 = graph.invoke(
#     {"messages": [{"role": "user", "content": "Patient MRN 21307 reports a mild headache after her last chemo session. No fever, vitals stable. Headache resolved with rest."}]},
#     config=config3
# )
# print("=" * 60)
# print("CASE 3: Simple Case - No Medication Change")
# print("=" * 60)
# for msg in result3["messages"]:
#     print(f"[{msg.type}]: {msg.content[:200]}")
#     print()
# print("Note: No physician approval needed -- case completed without interruption!")

print("Uncomment the code above to run Case 3.")

---
## Cell 5 -- Visualize the Multi-Agent Graph

In [ ]:
# TODO: Visualize the graph structure
# from IPython.display import Image, display
#
# try:
#     img_data = graph.get_graph().draw_mermaid_png()
#     display(Image(img_data))
# except Exception as e:
#     # Fallback: print the mermaid text representation
#     print("Graph structure (Mermaid format):")
#     print(graph.get_graph().draw_mermaid())

print("Uncomment the code above to visualize the graph.")

---
## Cell 6 -- Safety Guardrails

In a clinical system, we need guardrails to prevent runaway agent behavior and ensure auditability.

In [ ]:
# TODO: Add safety guardrails to the system

# Guardrail 1: Limit agent iterations to prevent infinite loops
# When creating specialist agents, set recursion_limit:
# result = graph.invoke(
#     {"messages": [...]},
#     config={"configurable": {"thread_id": "..."},  "recursion_limit": 10}
# )

# Guardrail 2: Audit logging -- track all tool calls
# def audit_log_node(state: MessagesState):
#     """Log all tool calls for clinical audit trail."""
#     from datetime import datetime
#     for msg in state["messages"]:
#         if hasattr(msg, 'tool_calls') and msg.tool_calls:
#             for tc in msg.tool_calls:
#                 print(f"[AUDIT {datetime.now().isoformat()}] Tool: {tc['name']}, Args: {tc['args']}")
#     return state

# Guardrail 3: Require human approval for ALL medication changes
# This is already built into our review_node!
# The key insight: interrupt() is a hard pause. The graph CANNOT continue
# without explicit human input. No timeout, no auto-approve.

# Guardrail 4: Prohibited actions list
# prohibited_actions = [
#     "discontinue_all_medications",
#     "override_allergy_alert",
#     "bypass_drug_interaction_warning",
# ]

print("Safety guardrails defined. In production, these would be enforced at every step.")
print("\nKey guardrails for clinical AI systems:")
print("  1. Iteration limits -- prevent runaway agent loops")
print("  2. Audit logging -- track every tool call for compliance")
print("  3. Human-in-the-loop -- interrupt() for medication changes")
print("  4. Prohibited actions -- hard-block dangerous operations")

---
## Stretch Goal: Add a Third Specialist (Palliative Care)

Add a palliative care specialist agent that is routed to when the patient's prognosis or stage suggests end-of-life considerations.

```python
# Hint: Add a new tool set for palliative care
# @tool
# def assess_symptom_burden(symptoms: str) -> str: ...
# @tool  
# def lookup_palliative_protocol(symptom: str) -> str: ...
#
# Then update the triage_node to include a third routing option:
# PALLIATIVE -> "palliative_node"
#
# And add the node to the graph:
# builder.add_node("palliative_node", palliative_node)
# builder.add_edge("palliative_node", END)
```

---
## KHCC Connection

This multi-agent pattern mirrors the **tumor board workflow** at KHCC where:
- A **triage coordinator** reviews incoming cases
- Cases are routed to the appropriate **specialist team** (surgical oncology, medical oncology, radiation oncology, palliative care)
- Any treatment plan change requires **senior oncologist approval** (our human-in-the-loop)
- All decisions are **logged for audit** and compliance

The `interrupt()` pattern is especially important in healthcare: no medication change should ever be auto-approved by an AI system. A physician must always be in the loop.